# 6. Genetic Algorithms

## 6.1 Introduction to Genetic Algorithms

Genetic Algorithms (GAs) are nature-inspired optimization algorithms that mimic the process of natural selection. Originally developed by John Holland in the 1970s, GAs have become powerful tools for solving complex optimization problems in various domains, including electrical machine design.

### 6.1.1 Biological Inspiration

Genetic algorithms are based on Darwin's theory of evolution:
- **Selection**: Fitter individuals are more likely to survive and reproduce
- **Crossover**: Genetic material is exchanged between parents to create offspring
- **Mutation**: Random changes occur in genetic material
- **Inheritance**: Offspring inherit characteristics from their parents

### 6.1.2 Key Components

1. **Population**: Set of potential solutions
2. **Chromosomes**: Representation of solutions (usually binary strings)
3. **Fitness Function**: Evaluates solution quality
4. **Genetic Operators**: Selection, crossover, and mutation

In [ ]:
import numpy as np
import random
from typing import List, Tuple, Callable

class GeneticAlgorithm:
    def __init__(self, 
                 population_size: int,
                 chromosome_length: int,
                 fitness_function: Callable,
                 mutation_rate: float = 0.01,
                 crossover_rate: float = 0.8):
        """Initialize Genetic Algorithm"""
        self.population_size = population_size
        self.chromosome_length = chromosome_length
        self.fitness_function = fitness_function
        self.mutation_rate = mutation_rate
        self.crossover_rate = crossover_rate
        self.population = self._initialize_population()
        
    def _initialize_population(self) -> np.ndarray:
        """Create initial random population"""
        return np.random.randint(0, 2, 
                                size=(self.population_size, self.chromosome_length))
    
    def evaluate_fitness(self) -> np.ndarray:
        """Evaluate fitness of all individuals"""
        return np.array([self.fitness_function(individual) 
                       for individual in self.population])
    
    def selection(self, fitness_scores: np.ndarray) -> np.ndarray:
        """Tournament selection"""
        selected = []
        for _ in range(self.population_size):
            tournament_size = min(3, self.population_size)
            tournament_indices = np.random.choice(self.population_size, 
                                                tournament_size, replace=False)
            tournament_fitness = fitness_scores[tournament_indices]
            winner_index = tournament_indices[np.argmax(tournament_fitness)]
            selected.append(self.population[winner_index].copy())
        return np.array(selected)
    
    def crossover(self, parent1: np.ndarray, parent2: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Single-point crossover"""
        if random.random() > self.crossover_rate:
            return parent1.copy(), parent2.copy()
        
        crossover_point = random.randint(1, self.chromosome_length - 1)
        child1 = np.concatenate([parent1[:crossover_point], parent2[crossover_point:]])
        child2 = np.concatenate([parent2[:crossover_point], parent1[crossover_point:]])
        return child1, child2
    
    def mutate(self, individual: np.ndarray) -> np.ndarray:
        """Bit flip mutation"""
        for i in range(len(individual)):
            if random.random() < self.mutation_rate:
                individual[i] = 1 - individual[i]
        return individual
    
    def evolve_generation(self) -> Tuple[np.ndarray, float, float]:
        """Evolve one generation"""
        # Evaluate fitness
        fitness_scores = self.evaluate_fitness()
        
        # Selection
        selected = self.selection(fitness_scores)
        
        # Crossover and mutation
        new_population = []
        for i in range(0, self.population_size, 2):
            parent1, parent2 = selected[i], selected[min(i+1, self.population_size-1)]
            child1, child2 = self.crossover(parent1, parent2)
            new_population.append(self.mutate(child1))
            new_population.append(self.mutate(child2))
        
        self.population = np.array(new_population[:self.population_size])
        
        return self.population, np.max(fitness_scores), np.mean(fitness_scores)

# Example usage
print("Genetic Algorithm class defined successfully")

## 6.2 Genetic Algorithms for Topology Optimization

In electrical machine topology optimization, GAs can effectively explore the complex design space of rotor and stator configurations.

### 6.2.1 Problem Encoding

For topology optimization of electrical machines:
- **Binary Encoding**: 1 = material, 0 = air/non-material
- **2D Grid Representation**: Discretized design domain
- **Chromosome Structure**: Flattened 2D topology representation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

class TopologyGeneticAlgorithm(GeneticAlgorithm):
    def __init__(self, grid_size: Tuple[int, int], **kwargs):
        self.grid_size = grid_size
        super().__init__(chromosome_length=grid_size[0] * grid_size[1], **kwargs)
    
    def chromosome_to_topology(self, chromosome: np.ndarray) -> np.ndarray:
        """Convert chromosome back to 2D topology"""
        return chromosome.reshape(self.grid_size)
    
    def topology_to_chromosome(self, topology: np.ndarray) -> np.ndarray:
        """Convert 2D topology to chromosome"""
        return topology.flatten()
    
    def visualize_topology(self, chromosome: np.ndarray, title: str = "Topology"):
        """Visualize topology as 2D grid"""
        topology = self.chromosome_to_topology(chromosome)
        
        plt.figure(figsize=(6, 6))
        plt.imshow(topology, cmap='binary', interpolation='nearest')
        plt.title(title)
        plt.colorbar(label='Material (1=Material, 0=Air)')
        plt.xlabel('X Position')
        plt.ylabel('Y Position')
        plt.grid(True, alpha=0.3)
        plt.show()
    
    def count_material_regions(self, chromosome: np.ndarray) -> int:
        """Count connected material regions"""
        topology = self.chromosome_to_topology(chromosome)
        visited = np.zeros_like(topology, dtype=bool)
        regions = 0
        
        def dfs(i, j):
            if (i < 0 or i >= self.grid_size[0] or 
                j < 0 or j >= self.grid_size[1] or 
                visited[i, j] or topology[i, j] == 0):
                return
            visited[i, j] = True
            dfs(i+1, j); dfs(i-1, j); dfs(i, j+1); dfs(i, j-1)
        
        for i in range(self.grid_size[0]):
            for j in range(self.grid_size[1]):
                if topology[i, j] == 1 and not visited[i, j]:
                    regions += 1
                    dfs(i, j)
        
        return regions

print("Topology GA class defined successfully")

### 6.2.2 Fitness Function for SynRM Design

For Synchronous Reluctance Machine (SynRM) design, the fitness function typically considers:

1. **Average Torque**: $\bar{T} = \frac{1}{2\pi} \int_0^{2\pi} T(\theta) d\theta$
2. **Torque Ripple**: $T_{ripple} = \frac{T_{max} - T_{min}}{\bar{T}}$
3. **Material Usage**: $\eta_{material} = \frac{\text{material pixels}}{\text{total pixels}}$
4. **Structural Integrity**: Penalty for disconnected regions

In [ ]:
class SynRMFitnessFunction:
    def __init__(self, grid_size: Tuple[int, int],
                 torque_weight: float = 1.0,
                 ripple_weight: float = 0.5,
                 material_weight: float = 0.3,
                 connectivity_weight: float = 0.2):
        self.grid_size = grid_size
        self.torque_weight = torque_weight
        self.ripple_weight = ripple_weight
        self.material_weight = material_weight
        self.connectivity_weight = connectivity_weight
    
    def calculate_torque_metrics(self, topology: np.ndarray) -> Tuple[float, float]:
        """Simplified torque calculation for demonstration"""
        # In practice, this would use FEA simulation
        # Here we use a simplified proxy based on flux barrier patterns
        
        material_fraction = np.sum(topology) / topology.size
        
        # Create a proxy for average torque based on material distribution
        # Higher torque for certain flux barrier patterns
        center_y, center_x = self.grid_size[0] // 2, self.grid_size[1] // 2
        
        # Calculate radial distribution
        y, x = np.ogrid[:self.grid_size[0], :self.grid_size[1]]
        distance_from_center = np.sqrt((y - center_y)**2 + (x - center_x)**2)
        
        # Proxy torque based on radial flux barrier effectiveness
        radial_distribution = np.sum(topology * distance_from_center) / (np.sum(topology) + 1e-6)
        avg_torque = material_fraction * (1.0 + 0.3 * np.sin(4 * radial_distribution / self.grid_size[1]))
        
        # Proxy torque ripple based on asymmetry
        asymmetry = np.abs(np.sum(topology[:self.grid_size[0]//2, :]) - 
                         np.sum(topology[self.grid_size[0]//2:, :]))
        torque_ripple = 0.1 + 0.2 * (asymmetry / (np.sum(topology) + 1e-6))
        
        return avg_torque, torque_ripple
    
    def calculate_connectivity_score(self, topology: np.ndarray) -> float:
        """Calculate connectivity penalty"""
        visited = np.zeros_like(topology, dtype=bool)
        regions = 0
        largest_region_size = 0
        
        def dfs(i, j, current_size):
            nonlocal largest_region_size
            if (i < 0 or i >= self.grid_size[0] or 
                j < 0 or j >= self.grid_size[1] or 
                visited[i, j] or topology[i, j] == 0):
                return
            
            visited[i, j] = True
            current_size[0] += 1
            largest_region_size = max(largest_region_size, current_size[0])
            
            dfs(i+1, j, current_size); dfs(i-1, j, current_size)
            dfs(i, j+1, current_size); dfs(i, j-1, current_size)
        
        for i in range(self.grid_size[0]):
            for j in range(self.grid_size[1]):
                if topology[i, j] == 1 and not visited[i, j]:
                    regions += 1
                    current_size = [0]
                    dfs(i, j, current_size)
        
        # Penalize multiple disconnected regions
        connectivity_score = 1.0 / (1.0 + 0.5 * regions)
        return connectivity_score
    
    def __call__(self, chromosome: np.ndarray) -> float:
        """Calculate fitness for SynRM topology"""
        topology = chromosome.reshape(self.grid_size)
        
        # Calculate torque metrics
        avg_torque, torque_ripple = self.calculate_torque_metrics(topology)
        
        # Calculate material usage efficiency
        material_fraction = np.sum(topology) / topology.size
        material_efficiency = 1.0 - material_fraction  # Less material is better
        
        # Calculate connectivity
        connectivity_score = self.calculate_connectivity_score(topology)
        
        # Combine objectives
        fitness = (self.torque_weight * avg_torque - 
                  self.ripple_weight * torque_ripple + 
                  self.material_weight * material_efficiency + 
                  self.connectivity_weight * connectivity_score)
        
        return fitness

print("SynRM fitness function defined successfully")

## 6.3 GA Experiment: SynRM Rotor Optimization

Let's run a complete GA experiment to optimize a SynRM rotor topology.

In [ ]:
# Set up the GA experiment
GRID_SIZE = (16, 16)  # 16x16 grid for rotor design
POPULATION_SIZE = 50
GENERATIONS = 100

# Initialize fitness function
fitness_func = SynRMFitnessFunction(
    grid_size=GRID_SIZE,
    torque_weight=1.0,
    ripple_weight=0.5,
    material_weight=0.3,
    connectivity_weight=0.2
)

# Initialize GA
ga = TopologyGeneticAlgorithm(
    grid_size=GRID_SIZE,
    population_size=POPULATION_SIZE,
    fitness_function=fitness_func,
    mutation_rate=0.02,
    crossover_rate=0.8
)

print(f"GA initialized with {POPULATION_SIZE} individuals for {GENERATIONS} generations")
print(f"Grid size: {GRID_SIZE[0]}x{GRID_SIZE[1]} (chromosome length: {ga.chromosome_length})")

In [ ]:
# Run evolution
evolution_history = {
    'best_fitness': [],
    'avg_fitness': [],
    'best_individual': None
}

for generation in range(GENERATIONS):
    population, max_fitness, avg_fitness = ga.evolve_generation()
    
    evolution_history['best_fitness'].append(max_fitness)
    evolution_history['avg_fitness'].append(avg_fitness)
    
    # Track best individual
    if evolution_history['best_individual'] is None or max_fitness > max(evolution_history['best_fitness']):
        best_idx = np.argmax([fitness_func(ind) for ind in population])
        evolution_history['best_individual'] = population[best_idx].copy()
    
    if generation % 10 == 0:
        print(f"Generation {generation:3d}: Best = {max_fitness:.4f}, Avg = {avg_fitness:.4f}")

print(f"\nEvolution completed! Best fitness: {max(evolution_history['best_fitness']):.4f}")

In [ ]:
# Visualize results
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Plot 1: Evolution history
axes[0, 0].plot(evolution_history['best_fitness'], label='Best Fitness', linewidth=2)
axes[0, 0].plot(evolution_history['avg_fitness'], label='Average Fitness', linewidth=2)
axes[0, 0].set_xlabel('Generation')
axes[0, 0].set_ylabel('Fitness')
axes[0, 0].set_title('GA Evolution History')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Best topology
best_topology = ga.chromosome_to_topology(evolution_history['best_individual'])
im1 = axes[0, 1].imshow(best_topology, cmap='binary', interpolation='nearest')
axes[0, 1].set_title('Best SynRM Rotor Topology')
axes[0, 1].set_xlabel('X Position')
axes[0, 1].set_ylabel('Y Position')
plt.colorbar(im1, ax=axes[0, 1], label='Material (1=Material, 0=Air)')

# Plot 3: Final population diversity
final_fitness = [fitness_func(ind) for ind in ga.population]
axes[1, 0].hist(final_fitness, bins=20, alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Fitness')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Final Population Fitness Distribution')
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Material usage statistics
material_usage = [np.sum(ind) / len(ind) for ind in ga.population]
axes[1, 1].hist(material_usage, bins=20, alpha=0.7, edgecolor='black', color='orange')
axes[1, 1].set_xlabel('Material Fraction')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Material Usage Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Print detailed statistics
print("\n=== GA Optimization Results ===")
print(f"Best fitness: {max(evolution_history['best_fitness']):.4f}")
print(f"Final average fitness: {evolution_history['avg_fitness'][-1]:.4f}")
print(f"Material usage in best solution: {np.sum(evolution_history['best_individual']) / len(evolution_history['best_individual']):.3f}")
print(f"Number of material regions: {ga.count_material_regions(evolution_history['best_individual'])}")

## 6.4 Advantages and Disadvantages of GAs for Topology Optimization

### 6.4.1 Advantages

1. **Global Search**: Can escape local optima better than gradient-based methods
2. **Discontinuity Handling**: Works well with discrete design variables
3. **Multi-objective**: Naturally handles multiple conflicting objectives
4. **Parallelizable**: Fitness evaluations can be computed in parallel
5. **No Derivatives Required**: Doesn't need gradient information

### 6.4.2 Disadvantages

1. **Computational Cost**: Requires many fitness function evaluations
2. **Parameter Tuning**: Performance sensitive to GA parameters
3. **Slow Convergence**: May require many generations to converge
4. **No Guarantee**: Cannot guarantee global optimum
5. **Representation Dependency**: Solution quality depends on encoding scheme

## 6.5 Genetic Algorithms vs. MDP Approach

| Aspect | Genetic Algorithms | MDP/Reinforcement Learning |
|--------|------------------|----------------------------|
| **Search Strategy** | Population-based evolutionary | Sequential decision-making |
| **Exploration** | Mutation & crossover operators | ε-greedy, softmax policies |
| **Exploitation** | Selection pressure | Value function maximization |
| **Memory** | Population preserves good solutions | Q-table or neural network memory |
| **Convergence** | Statistical convergence over generations | Policy convergence over episodes |
| **Parallelism** | Naturally parallel | Typically sequential |
| **Parameter Sensitivity** | High (mutation, crossover rates) | Moderate (learning rate, discount) |
| **Local Optima** | Better at escaping | Can get stuck in local optima |

### 6.5.1 Hybrid Approaches

Recent research explores combining GAs with RL:
- **GA for initialization**: Use GA to generate initial population, then refine with RL
- **RL for parameter adaptation**: Use RL to adapt GA parameters during evolution
- **Multi-objective optimization**: Combine both methods for complex trade-offs

## 6.6 Conclusions

Genetic Algorithms provide a powerful approach to topology optimization for electrical machines:

### 6.6.1 Key Takeaways

1. **Effectiveness**: GAs can discover non-intuitive optimal topologies
2. **Flexibility**: Easily handles multiple objectives and constraints
3. **Robustness**: Works with discontinuous and non-differentiable fitness landscapes
4. **Scalability**: Performance scales with available computational resources

### 6.6.2 Future Directions

1. **Adaptive GAs**: Self-adjusting parameters based on search progress
2. **Hybrid Methods**: Combining GAs with gradient-based optimization
3. **Multi-fidelity**: Using approximate models for initial screening
4. **Real-world Integration**: Manufacturing constraints and uncertainty quantification

### 6.6.3 Practical Recommendations

For SynRM topology optimization using GAs:
- Start with moderate population sizes (50-100)
- Use adaptive mutation rates
- Incorporate domain knowledge in fitness function
- Validate results with high-fidelity simulations
- Consider hybrid approaches for large-scale problems